 **O que este script faz?**

 1. **Ingestão (Extract):** Coleta dados de filmes de três fontes distintas (arquivos CSV no GitHub, tabelas na nuvem via Google BigQuery e APIs externas).

 2. **Transformação (Transform):** Cruza tabelas (Merge/Join), limpa strings, remove duplicatas e manipula listas para identificar atores de destaque da Marvel e DC.

 3. **Enriquecimento:** Utiliza a API do TMDB para converter nomes de atores em IDs e, em seguida, em arrobas de Instagram. Usa Web Scraping/API não oficial para coletar métricas de redes sociais.

 4. **Carga (Load):** Salva os resultados estruturados de volta na Nuvem (Google BigQuery) em uma camada chamada "Bronze" (dados brutos armazenados).

In [ ]:
# ==============================================================================
# 1. IMPORTAÇÃO DE BIBLIOTECAS E DEPENDÊNCIAS
# ==============================================================================

# Ferramentas de Nuvem (Google Cloud Platform - GCP)
from google.cloud import bigquery  # Conector oficial do banco de dados analítico (BigQuery) da Google.
from google.oauth2 import service_account  # Mecanismo de autenticação por chave/arquivo JSON de conta de serviço.

# Manipulação e Análise de Dados
import pandas as pd  # A principal biblioteca de Data Science para tabelas (DataFrames) e manipulação matricial.

In [ ]:
# ==============================================================================
# 2. CONFIGURAÇÕES DE VISUALIZAÇÃO (PANDAS DISPLAY CONFIG)
# ==============================================================================

# Por padrão, o Pandas esconde colunas/linhas se a tabela for muito grande.
# Estas configurações garantem que possamos inspecionar os dados no terminal/notebook.
pd.set_option("display.max_columns", None)  # Não limita o número máximo de colunas exibidas na tela.
pd.set_option("display.max_rows", 10)  # Limita a exibição visual a no máximo 10 linhas para não travar a IDE.

In [ ]:
# ==============================================================================
# 3. CONFIGURAÇÃO DE FONTES DE DADOS E INFRAESTRUTURA DE NUVEM
# ==============================================================================

# --- FONTE 1: REPOSITÓRIO REMOTO (GITHUB) ---
# URL base onde estão armazenados arquivos CSV públicos contendo dados históricos de filmes.
url_github_movies_dataset = "https://media.githubusercontent.com/media/lucas-aulas/dataset-movies/refs/heads/main"

# --- FONTE 2: AMBIENTE DE LEITURA (PROJETO COMPARTILHADO DO PROFESSOR) ---
# ID do projeto na Google Cloud de onde leremos uma tabela institucional de bilheterias.
fonte_gcp_project_id = "_______________________"

# --- DESTINO: AMBIENTE DE ESCRITA (PROJETO PRIVADO DO ALUNO) ---
# Caminho local do arquivo JSON que contém as credenciais de segurança (chave privada).
# ATENÇÃO: Nunca envie esta chave para o GitHub público! Ela equivale à senha do seu banco de dados.
meu_service_account_file = "../secrets/_______________________"
meu_gcp_project_id = "_______________________"  # ID do seu projeto pessoal criado no GCP.

# Inicialização da autenticação e do cliente BigQuery
# Transforma o arquivo JSON em um objeto de credenciais aceito pelo Google.
gcp_credentials = service_account.Credentials.from_service_account_file(meu_service_account_file)

# O objeto 'client' funciona como a nossa ponte ativa de conexão/comunicação com a nuvem do Google.
bigquery_client = bigquery.Client(credentials=gcp_credentials)

In [ ]:
# ==============================================================================
# 4. EXTRAÇÃO DE DADOS (EXTRACT)
# ==============================================================================

# Leitura de um arquivo CSV hospedado na web diretamente para a memória como um DataFrame do Pandas.
# Este dataset contém dados gerais de filmes avaliados no site IMDB (Internet Movie Database).
df_imdb_filmes = pd.read_csv(f"{url_github_movies_dataset}/imdb_filmes.csv")

In [ ]:
# Execução de uma consulta SQL estruturada diretamente dentro do Data Warehouse (BigQuery) da nuvem.
# O método '.to_dataframe()' faz o download do resultado da query SQL e o converte em uma tabela Pandas local.
# 'create_bqstorage_client=False' evita criar uma conexão otimizada adicional, mantendo a chamada simples.
df_boxoffice_dc_marvel = bigquery_client.query(
    f"SELECT * FROM {fonte_gcp_project_id}.raw.boxoffice_dc_marvel"
).to_dataframe(create_bqstorage_client=False)

In [ ]:
# ==============================================================================
# 5. SANITIZAÇÃO DE METADADOS (ORDENAÇÃO DE COLUNAS)
# ==============================================================================

# Boas práticas em Engenharia de Dados: Garantir previsibilidade visual.
# Aqui reordenamos as colunas de todas as tabelas em ordem alfabética (A-Z).
# Exemplo: Se as colunas fossem ['B', 'A', 'C'], viram ['A', 'B', 'C'].
df_boxoffice_dc_marvel = df_boxoffice_dc_marvel[sorted(df_boxoffice_dc_marvel.columns)]
df_imdb_filmes = df_imdb_filmes[sorted(df_imdb_filmes.columns)]

In [ ]:
# ==============================================================================
# 6. ANÁLISE EXPLORATÓRIA PRELIMINAR (DATA EXPLORATION)
# ==============================================================================

# Filtragens de teste para compreender a estrutura e consistência do nome "Hulk" nas 3 fontes.
# '.head(1)' retorna apenas o primeiro registro encontrado, servindo como uma amostragem.

In [ ]:
df_boxoffice_dc_marvel[df_boxoffice_dc_marvel["FILM"] == "Hulk"].head(1)

In [ ]:
df_imdb_filmes[df_imdb_filmes["TITLE"] == "Hulk"].head(1)

In [ ]:
# ==============================================================================
# 7. CARGA INICIAL - CAMADA BRONZE (LOAD)
# ==============================================================================

# Enviando os dados brutos locais para tabelas físicas dentro do BigQuery.
# 'destination_table="bronze.nome"' cria ou insere os dados no Dataset chamado 'bronze'.
# 'if_exists="replace"' indica que se a tabela já existir lá em cima, ela será apagada e recriada do zero.
df_imdb_filmes.to_gbq(
    project_id=meu_gcp_project_id,
    destination_table="bronze.imdb_filmes",
    if_exists="replace",
    credentials=gcp_credentials,
)

df_boxoffice_dc_marvel.to_gbq(
    project_id=meu_gcp_project_id,
    destination_table="bronze.boxoffice_dc_marvel",
    if_exists="replace",
    credentials=gcp_credentials,
)